# Практика: как K-means движется между двумя облаками

Работайте сверху вниз. Нужны `numpy`, `pandas`, `matplotlib` и `scikit-learn`. Учебные 22 точки повторяют пример из текста; набор Iris входит в библиотеку.

Сначала предскажите, сколько раундов потребуется, если оба центра находятся справа. Затем сравните с началом, где по одному центру в каждом видимом облаке. Номера групп условны.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris

coordinates = [(1, 1), (2, 1), (1, 2), (2, 2), (2, 3), (3, 3),
               (1, 4), (2, 4), (3, 4), (1, 5), (2, 5), (2, 6),
               (7, 6), (9, 6), (11, 6), (13, 6), (8, 7),
               (10, 7), (12, 7), (14, 7), (9, 8), (13, 8)]
points = pd.DataFrame(coordinates, columns=['x', 'y'])
points.insert(0, 'id', list('ABCDEFGHIJKLMNOPQRSTUV'))
coords = points[['x', 'y']].to_numpy(dtype=float)
print(points.to_string(index=False))


## 1. Пять раундов от неудобного старта

Сначала проверьте расстояния от A, M и V до центров (13;6) и (14;7). В таблице каждый раунд означает распределение всех точек и пересчёт средних. Последний раунд проверяет, что группы не изменились.


In [ ]:
centers = np.array([[13., 6.], [14., 7.]])
previous = None
history = []
snapshots = [{'labels': None, 'centers': centers.copy(), 'moved': []}]
for number in range(1, 21):
    squared = ((coords[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    labels = squared.argmin(axis=1)
    groups = [coords[labels == group] for group in range(2)]
    if any(len(group) == 0 for group in groups):
        raise ValueError('Одна группа пуста')
    new_centers = np.array([group.mean(axis=0) for group in groups])
    moved = [] if previous is None else points.loc[labels != previous, 'id'].tolist()
    J = ((coords - new_centers[labels]) ** 2).sum()
    snapshots.append({'labels': labels.copy(), 'centers': new_centers.copy(), 'moved': moved})
    history.append({'раунд': number, 'размеры': [len(group) for group in groups],
                    'перешли': moved, 'c1': new_centers[0].round(2).tolist(),
                    'c2': new_centers[1].round(2).tolist(), 'J': round(float(J), 2)})
    if previous is not None and np.array_equal(labels, previous):
        break
    centers, previous = new_centers, labels
print(pd.DataFrame(history).to_string(index=False))
assert len(history) == 5
assert np.array_equal(labels, [0] * 12 + [1] * 10)


**Самостоятельно.** Уберите из правого облака M и запустите ячейку заново. Поменяется ли число раундов? Затем верните M и сравните другой старт: A (1;1) и V (13;8). Сохраняйте одинаковые точки, когда сравниваете значения J.

**Самопроверка для исходных 22 точек.** Размеры групп идут как 20/2, 15/7, 13/9, 12/10, 12/10. В пятом раунде никто не переходит.


In [ ]:
starts = {'оба справа': [[13, 6], [14, 7]],
          'по одному в облаке': [[1, 1], [13, 8]]}
for name, initial in starts.items():
    model = KMeans(n_clusters=2, init=np.array(initial, dtype=float),
                   n_init=1, tol=0, algorithm='lloyd').fit(coords)
    print(name, 'раундов:', model.n_iter_, 'J:', round(model.inertia_, 2),
          'размеры:', np.bincount(model.labels_).tolist())

fig, axes = plt.subplots(3, 2, figsize=(12, 11), sharex=True, sharey=True)
colors = ['#2459c2', '#b05300']
for number, (ax, state) in enumerate(zip(axes.flat, snapshots)):
    current = state['centers']
    if number == 0:
        ax.scatter(coords[:, 0], coords[:, 1], color='#1b2430', s=24)
        ax.set_title('Начало: оба центра справа')
        for name in ['A', 'M', 'O', 'V']:
            row = points.loc[points['id'] == name].iloc[0]
            ax.annotate(name, (row['x'], row['y']), xytext=(6, 5),
                        textcoords='offset points')
    else:
        labels_now = state['labels']
        for group, color in enumerate(colors):
            part = coords[labels_now == group]
            ax.scatter(part[:, 0], part[:, 1], color=color, s=24,
                       label=f'Группа {group + 1}')
        if state['moved']:
            changed = points['id'].isin(state['moved']).to_numpy()
            ax.scatter(coords[changed, 0], coords[changed, 1],
                       facecolors='none', edgecolors='#7238c4', s=105, linewidths=2)
        before = snapshots[number - 1]['centers']
        for group, color in enumerate(colors):
            ax.plot([before[group, 0], current[group, 0]],
                    [before[group, 1], current[group, 1]],
                    color=color, linestyle='--', linewidth=1.5)
            if not np.allclose(before[group], current[group]):
                ax.scatter(*before[group], marker='D', facecolors='white',
                           edgecolors=color, s=90, linewidths=1.5)
        ax.set_title(f'Раунд {number}: {np.bincount(labels_now).tolist()}')
        moved_text = ', '.join(state['moved']) if state['moved'] else 'никто'
        ax.text(.5, -.18, f'Перешли: {moved_text}', transform=ax.transAxes,
                ha='center', fontsize=10)
    for group, color in enumerate(colors):
        ax.scatter(*current[group], marker='D', color=color, s=100,
                   edgecolors='white', linewidths=.7)
    ax.set(xlim=(0, 15), ylim=(0, 9), xlabel='x, усл. м', ylabel='y, усл. м')
    ax.set_aspect('equal', adjustable='box')
    ax.grid(alpha=.25)
fig.subplots_adjust(hspace=.45, wspace=.18)
plt.show()


## 2. Ирисы: таблица и диаграмма

Модель увидит только длину и ширину лепестка. Известный вид нужен для интерпретации уже полученных групп, но не для обучения. На диаграмме окраска показывает найденный кластер.


In [ ]:
iris = load_iris()
flowers = pd.DataFrame(iris.data[:, 2:4], columns=['length_cm', 'width_cm'])
initial = flowers.iloc[[0, 50, 100]].to_numpy()
model = KMeans(n_clusters=3, init=initial, n_init=1, random_state=0)
flowers['cluster'] = model.fit_predict(flowers[['length_cm', 'width_cm']])
print(flowers.groupby('cluster')[['length_cm', 'width_cm']].mean().round(2))
fig, ax = plt.subplots(figsize=(7, 4))
for group, part in flowers.groupby('cluster'):
    ax.scatter(part['length_cm'], part['width_cm'], s=18, alpha=.7, label=f'Группа {group}')
ax.scatter(model.cluster_centers_[:, 0], model.cluster_centers_[:, 1],
           marker='X', s=130, c='black', label='Центры')
ax.set(xlabel='Длина лепестка, см', ylabel='Ширина лепестка, см', title='Ирисы после K-means')
ax.legend()
plt.show()


**Самостоятельно.** Возьмите другие три строки для начальных центров, например `flowers.iloc[[10, 60, 110]][['length_cm', 'width_cm']].to_numpy()`. Постройте модель заново. Запишите сумму квадратов расстояний, размеры групп и положение центров. Что поменялось? Если численные номера групп поменялись, сравните также их состав, а не только номера.

**Дополнительно.** Попробуйте `KMeans(n_clusters=3, n_init=10, random_state=0)` и сравните `inertia_`. Почему меньшая сумма не доказывает совпадения групп с видами?

**Самопроверка.** Проверяйте, что `len(flowers) == 150`, суммарный размер групп равен 150 и виды не входят в признаки. `inertia_` оценивает компактность групп по двум числовым признакам; биологический смысл она не измеряет.


## 3. Что меняется при другом расстоянии

Здесь мы сохраняем пересчёт центров как средних и меняем только способ находить ближайший центр. При равенстве `np.argmin` выбирает первый. Сравните траектории с таблицей 4 в тексте. Это опыт на выбранных точках, а не доказательство остановки для любых данных.


In [ ]:
def rounds_with_mean(norm, max_rounds=20):
    centers = np.array([[13., 6.], [14., 7.]])
    previous = None
    seen = set()
    sizes = []
    for _ in range(max_rounds):
        delta = np.abs(coords[:, None, :] - centers[None, :, :])
        if norm == 'L2':
            distance = np.sqrt((delta ** 2).sum(axis=2))
        elif norm == 'L1':
            distance = delta.sum(axis=2)
        else:
            distance = delta.max(axis=2)
        labels = distance.argmin(axis=1)  # при равенстве — первый центр
        signature = tuple(labels.tolist())
        if signature in seen and not np.array_equal(labels, previous):
            raise RuntimeError('Распределение повторилось без остановки')
        seen.add(signature)
        groups = [coords[labels == group] for group in range(2)]
        if any(len(group) == 0 for group in groups):
            raise ValueError('Одна группа пуста')
        new_centers = np.array([group.mean(axis=0) for group in groups])
        sizes.append(tuple(len(group) for group in groups))
        if previous is not None and np.array_equal(labels, previous):
            return sizes, new_centers
        centers, previous = new_centers, labels.copy()
    raise RuntimeError('Достигнут предел раундов')

for norm in ('L2', 'L1', 'Linf'):
    sizes, centers_for_norm = rounds_with_mean(norm)
    print(norm, sizes, centers_for_norm.round(2).tolist())
assert [len(rounds_with_mean(norm)[0]) for norm in ('L2', 'L1', 'Linf')] == [5, 4, 5]


## Самопроверка




### 1. Что известно до запуска кластеризации в нашем примере?

A. Правильная группа каждого места.

B. Координаты мест и выбранное число групп $K=2$.

C. Только готовые центры групп.

D. Эталонный индекс Рэнда.




### 2. Что означает $K=2$ в K-means?

A. У каждой точки две координаты.

B. Алгоритм всегда делает ровно два шага.

C. Мы ищем две группы.

D. В каждой группе должно быть по две точки.




### 3. Какой квадрат расстояния от A $(1;1)$ до начального центра $(13;6)$?

A. 25.

B. 144.

C. 169.

D. 205.




### 4. При центрах $(13;6)$ и $(14;7)$ в какую группу попадёт M $(7;6)$?

A. К первому центру: квадраты расстояний 36 и 50.

B. Ко второму центру: он находится в том же видимом облаке.

C. Сразу в обе группы.

D. Ни в одну: центры ещё не обновлены.




### 5. Каким станет первый центр после первого распределения 20 и 2 пунктов?

A. $(13;6)$.

B. $(5{,}05;4{,}65)$.

C. $(13{,}5;7{,}5)$.

D. $(1{,}83;3{,}33)$.




### 6. Что произойдёт на втором раунде с пунктом O $(11;6)$?

A. Он останется в первой группе, потому что там был раньше.

B. Он перейдёт во вторую группу: она теперь ближе.

C. Он войдёт сразу в обе группы.

D. Он станет новым центром без пересчёта остальных пунктов.




### 7. Для точек $(0;0)$, $(0;0)$, $(9;9)$ какой вывод верен?

A. Среднее $(3;3)$ лучше для суммы квадратов $L_2$, а $(0;0)$ даёт меньшую сумму $L_1$.

B. Среднее $(3;3)$ даёт наименьшую сумму $L_1$.

C. Для суммы квадратов $L_2$ лучший центр — $(0;0)$.

D. При любом расстоянии центром обязательно должна быть одна из исходных точек.




### 8. Какова итоговая сумма квадратов расстояний $J$ для двух облаков?

A. 30,33.

B. 90,33.

C. 173,30.

D. 430,50.




### 9. Неудобный и удачный старты дали одинаковые группы. Какой вывод верен для этих точек?

A. Все возможные старты всегда дают те же группы.

B. Начальные центры изменили число раундов: пять и два.

C. Пять раундов дали обязательно лучшее разбиение, чем два.

D. Теперь номера групп стали правильными метками.




### 10. Когда при кластеризации ирисов можно использовать известные названия видов?

A. Добавить вид как ещё один признак перед запуском K-means.

B. Подменить видом номер ближайшего центра при распределении цветков.

C. Сравнить виды с полученными группами после запуска.

D. Назвать группы видами только по их номерам.




## Подсказки и решения

### Вопрос 1

**Подсказка**


Правильных меток у мест нет.



**Решение**


**B.** Координаты заданы, а две группы нужны по условию задачи. Группы и их центры ещё предстоит найти.



### Вопрос 2

**Подсказка**


Не путайте число координат и число групп.



**Решение**


**C.** $K$ задаёт число групп. Число шагов и размеры групп могут быть другими.



### Вопрос 3

**Подсказка**


Возведите в квадрат разности обеих координат и сложите.



**Решение**


**C.** $(1-13)^2+(1-6)^2=144+25=169$. Нужно учесть обе координаты.



### Вопрос 4

**Подсказка**


Сравните расстояния, не пользуясь видимой формой облаков.



**Решение**


**A.** M ближе к первому центру, поскольку $36<50$. Видимое облако пока не совпало с группой алгоритма.



### Вопрос 5

**Подсказка**


Суммы координат первой группы равны 101 и 93; в группе 20 пунктов.



**Решение**


**B.** Средние координаты равны $101/20=5{,}05$ и $93/20=4{,}65$.



### Вопрос 6

**Подсказка**


Сравните расстояния до $(5{,}05;4{,}65)$ и $(13{,}5;7{,}5)$.



**Решение**


**B.** Квадраты расстояний примерно $37{,}23$ и $8{,}5$. O меняет группу.



### Вопрос 7

**Подсказка**


Сравните суммы при центрах $(3;3)$ и $(0;0)$ для квадрата $L_2$ и для $L_1$.



**Решение**


**A.** Для суммы квадратов получаем $108<162$, а для суммы $L_1$ — $24>18$. Лучший центр зависит от того, какую сумму мы уменьшаем.



### Вопрос 8

**Подсказка**


Сложите вклады двух групп: примерно 30,33 и 60.



**Решение**


**B.** Вклады итоговых групп составляют примерно $30{,}33$ и $60$, вместе $90{,}33$.



### Вопрос 9

**Подсказка**


Сравните таблицу пяти раундов и запуск с центрами A и V.



**Решение**


**B.** На этих точках итог совпал, но число раундов различалось. Для другого набора данных могут различаться и сами устойчивые группировки.



### Вопрос 10

**Подсказка**


На каком шаге нашего примера мы открыли названия видов?



**Решение**


**C.** K-means получал только измерения цветков. Известные виды можно открыть после группировки и сравнить с ней; если передать их алгоритму как признак, это уже не будет группировкой без правильных ответов.



